In [1]:
import cv2
import numpy as np

In [ ]:
camera = cv2.VideoCapture(0)

faces_cascade = cv2.CascadeClassifier("C:\\Users\\bhara\\Downloads\\haarcascade_frontalface_default.xml")
eyes_cascade = cv2.CascadeClassifier("C:\\Users\\bhara\\Downloads\\haarcascade_eye.xml")
smile_cascade = cv2.CascadeClassifier("C:\\Users\\bhara\\Downloads\\haarcascade_smile.xml")

if not camera.isOpened():
    print("Error: Camera not working")
    exit()

while True:
    ret, frame = camera.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = faces_cascade.detectMultiScale(gray, 1.2, 6, minSize=(80, 80))

    if len(faces) == 0:
        cv2.putText(frame, "No Face Detected", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

    status = "No Face Detected"
    for (x,y,w,h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, "Face Detected", (x, y-20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eyes_cascade.detectMultiScale(roi_gray, 1.2, 20, minSize=(20, 20))

        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0 , 0), 2)
            cv2.putText(roi_color, "Eye", (ex, ey-2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 1)

        valid_smiles = []
        smiles = smile_cascade.detectMultiScale(roi_gray, 2, 50)

        for(sx, sy, sw, sh) in smiles:
            if sw > w * 0.35 and sy > h * 0.5:   # only accept big smiles
                valid_smiles.append([sx, sy, sw, sh])
                cv2.rectangle(roi_color, (sx, sy), (sx+sw, sy+sh), (0,255,0), 2)
                cv2.putText(roi_color, "Smiling", (sx, sy-2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 1)

        if len(valid_smiles) > 0:
            status = "You are Happy!"
        elif len(eyes) == 0:
            status = "You are Drowsy!"
        else:
            status = "You are Active!"

    cv2.putText(frame, status, (20, 80), cv2.FONT_HERSHEY_COMPLEX, 1, (255,0,255), 2)
    cv2.imshow("Smart Face Monitor", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()